In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "devops-t13-qa"
NOTEBOOK_PATH = "notebooks/qa/43-devops-t13-qa.ipynb"
TOWER = 13
TOWER_NAME = "Tower of DevOps"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 13: Tower of DevOps


In [2]:
# Cell 1: CI/CD Configuration (Floors 1-7)

workflows_dir = BROWSER_ROOT / ".github" / "workflows"

# P1: CI workflow file exists
ci_yml = read_file(workflows_dir / "ci.yml")
has_ci = len(ci_yml) > 50
record("P1", "CI workflow exists (.github/workflows/ci.yml)", has_ci,
       f"CI config: {len(ci_yml)} chars")

# P2: CI triggers on push
has_push_trigger = 'push:' in ci_yml and 'branches:' in ci_yml
record("P2", "CI triggers on push to all branches", has_push_trigger,
       "Push trigger configured with branch filter")

# P3: CI triggers on pull_request
has_pr_trigger = 'pull_request' in ci_yml
record("P3", "CI triggers on pull_request", has_pr_trigger,
       "PR trigger configured for code review")

# P4: CI has lint job
has_lint_job = 'lint:' in ci_yml.lower() or 'Lint' in ci_yml
record("P4", "CI has lint job", has_lint_job,
       "Ruff linting in CI pipeline")

PASS: CI workflow exists (.github/workflows/ci.yml) -- CI config: 6154 chars
PASS: CI triggers on push to all branches -- Push trigger configured with branch filter
PASS: CI triggers on pull_request -- PR trigger configured for code review
PASS: CI has lint job -- Ruff linting in CI pipeline


In [3]:
# Cell 2: CI Test and Release Jobs (Floors 8-14)

# P5: CI has test job with matrix strategy
has_test_job = 'test:' in ci_yml and 'matrix' in ci_yml
record("P5", "CI has test job with Python matrix", has_test_job,
       "Tests run on multiple Python versions")

# P6: CI runs pytest
has_pytest_run = 'pytest' in ci_yml
record("P6", "CI runs pytest", has_pytest_run,
       "pytest command found in CI workflow")

# P7: CI has release job (triggered on tags)
has_release = 'release:' in ci_yml and 'tags' in ci_yml
record("P7", "CI has release job (tag-triggered)", has_release,
       "Release job triggers on vX.Y.Z tags")

# P8: CI computes SHA256 for release binaries
has_sha256 = 'sha256' in ci_yml.lower() or 'SHA256' in ci_yml
record("P8", "CI computes SHA256 for release binaries", has_sha256,
       "Binary integrity verification in release pipeline")

PASS: CI has test job with Python matrix -- Tests run on multiple Python versions
PASS: CI runs pytest -- pytest command found in CI workflow
PASS: CI has release job (tag-triggered) -- Release job triggers on vX.Y.Z tags
PASS: CI computes SHA256 for release binaries -- Binary integrity verification in release pipeline


In [4]:
# Cell 3: Dockerfile Structure (Floors 15-21)

dockerfile = read_file(BROWSER_ROOT / "Dockerfile")

# P9: Dockerfile exists
has_dockerfile = len(dockerfile) > 50
record("P9", "Dockerfile exists", has_dockerfile,
       f"Dockerfile: {len(dockerfile)} chars")

# P10: Multi-stage build
has_multistage = dockerfile.count('FROM ') >= 2
record("P10", "Dockerfile uses multi-stage build", has_multistage,
       f"FROM statements: {dockerfile.count('FROM ')}")

# P11: Non-root user configured
has_nonroot = 'USER solace' in dockerfile or 'USER nonroot' in dockerfile
record("P11", "Dockerfile runs as non-root user", has_nonroot,
       "Security: container runs as unprivileged user")

# P12: Health check configured
has_healthcheck = 'HEALTHCHECK' in dockerfile
record("P12", "Dockerfile has HEALTHCHECK directive", has_healthcheck,
       "Container health monitoring configured")

PASS: Dockerfile exists -- Dockerfile: 3880 chars
PASS: Dockerfile uses multi-stage build -- FROM statements: 2
PASS: Dockerfile runs as non-root user -- Security: container runs as unprivileged user
PASS: Dockerfile has HEALTHCHECK directive -- Container health monitoring configured


In [5]:
# Cell 4: Cloud Deploy and Build Config (Floors 22-28)

# P13: cloud-run-deploy.yaml exists
cloud_deploy = read_file(BROWSER_ROOT / "cloud-run-deploy.yaml")
has_cloud_deploy = len(cloud_deploy) > 50
record("P13", "cloud-run-deploy.yaml exists", has_cloud_deploy,
       f"Cloud Run deploy config: {len(cloud_deploy)} chars")

# P14: Cloud Run has health probes configured
has_probes = 'livenessProbe' in cloud_deploy and 'readinessProbe' in cloud_deploy
record("P14", "Cloud Run has liveness and readiness probes", has_probes,
       "Health probes ensure container is serving")

# P15: Dockerfile sets PORT environment variable
has_port = 'PORT=8080' in dockerfile or 'ENV PORT' in dockerfile
record("P15", "Dockerfile sets PORT environment variable", has_port,
       "PORT=8080 configured for Cloud Run")

# P16: Dockerfile EXPOSE directive
has_expose = 'EXPOSE 8080' in dockerfile
record("P16", "Dockerfile has EXPOSE 8080", has_expose,
       "Container port exposed for Cloud Run")

PASS: cloud-run-deploy.yaml exists -- Cloud Run deploy config: 5377 chars
PASS: Cloud Run has liveness and readiness probes -- Health probes ensure container is serving
PASS: Dockerfile sets PORT environment variable -- PORT=8080 configured for Cloud Run
PASS: Dockerfile has EXPOSE 8080 -- Container port exposed for Cloud Run


In [6]:
# Cell 5: Test Infrastructure (Floors 29-35)

# P17: pyproject.toml has pytest config
pyproject = read_file(BROWSER_ROOT / "pyproject.toml")
has_pytest_config = '[tool.pytest' in pyproject
record("P17", "pyproject.toml has pytest configuration", has_pytest_config,
       "Test markers and options configured")

# P18: requirements.txt exists (Python dependencies)
has_requirements = (BROWSER_ROOT / "requirements.txt").exists()
record("P18", "requirements.txt exists (Python deps)", has_requirements,
       "Python dependency list for reproducible builds")

# P19: package-lock.json exists (Node.js lockfile)
has_pkg_lock = (BROWSER_ROOT / "package-lock.json").exists()
record("P19", "package-lock.json exists (Node.js lockfile)", has_pkg_lock,
       "Node.js dependency versions locked")

# P20: .dockerignore exists
has_dockerignore = (BROWSER_ROOT / ".dockerignore").exists()
record("P20", ".dockerignore exists", has_dockerignore,
       "Docker build excludes unnecessary files")

PASS: pyproject.toml has pytest configuration -- Test markers and options configured
PASS: requirements.txt exists (Python deps) -- Python dependency list for reproducible builds
PASS: package-lock.json exists (Node.js lockfile) -- Node.js dependency versions locked
PASS: .dockerignore exists -- Docker build excludes unnecessary files


In [7]:
# Cell 6: Additional Workflows and Build Scripts (Floors 36-49)

# P21: Build-binaries workflow exists
has_build_binaries = (workflows_dir / "build-binaries.yml").exists()
record("P21", "build-binaries.yml workflow exists", has_build_binaries,
       "Binary build pipeline for all platforms")

# P22: Web architecture workflow exists
has_web_arch = (workflows_dir / "web-architecture.yml").exists()
record("P22", "web-architecture.yml workflow exists", has_web_arch,
       "Web architecture validation in CI")

# P23: VERSION file exists (build versioning)
version_file = read_file(BROWSER_ROOT / "VERSION")
has_version = len(version_file.strip()) > 0
record("P23", "VERSION file exists", has_version,
       f"Version: {version_file.strip()[:20]}")

# P24: CI tests across multiple Python versions
python_versions = re.findall(r'python-version.*?\[([^\]]+)\]', ci_yml, re.DOTALL)
# Also check matrix: directly
has_multi_py = '3.11' in ci_yml and '3.12' in ci_yml
record("P24", "CI tests multiple Python versions (3.11, 3.12)", has_multi_py,
       "Tests validated on Python 3.11 and 3.12")

# P25: CI uses GitHub Actions cache for pip
has_cache = 'cache: "pip"' in ci_yml or "cache: 'pip'" in ci_yml or 'cache: pip' in ci_yml
record("P25", "CI uses pip cache for faster builds", has_cache,
       "Dependency caching speeds up CI")

PASS: build-binaries.yml workflow exists -- Binary build pipeline for all platforms
PASS: web-architecture.yml workflow exists -- Web architecture validation in CI
PASS: VERSION file exists -- Version: 1.1.0
PASS: CI tests multiple Python versions (3.11, 3.12) -- Tests validated on Python 3.11 and 3.12
PASS: CI uses pip cache for faster builds -- Dependency caching speeds up CI


In [8]:
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 13: Tower of DevOps
Probes: 25 | Passed: 25 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: c81355b3f2310fbe

{
  "surface": "devops-t13-qa",
  "tower": 13,
  "tower_name": "Tower of DevOps",
  "timestamp": "2026-03-06T11:28:52.264685",
  "total_probes": 25,
  "passed": 25,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "c81355b3f2310fbe"
}
